# Feature Selection Evaluation and Comparison

This notebook performs an honest, head-to-head comparison of different feature selection strategies for Spatio-temporal soil moisture modeling on the Washington-only `derived_8.1_pos` dataset:
- **Set A**: `MDR-v25` baseline hand-selected features (38 features, including elevation, slope, and seasonal variables).
- **Set B**: Default automatically selected features (40 features, selected using the default MI=120 pipeline).
- **Set C**: ElasticNet-only selection (No Mutual Information pre-filter).
- **Set D**: High MI limit (`k=300` pre-filter before ElasticNet).
- **Set E**: Hybrid Selection (bypassing MI for static/seasonal/precipitation features, and only using MI to filter rolling features).

We train a global XGBoost model (re-weighted with recency weights, $\beta=0.2$) and evaluate $R^2$, RMSE, and Pearson correlation on the test split.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Find and configure project root
def find_project_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for cand in candidates:
        if (cand / "data").exists() and (cand / "Modeling").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from Modeling.Utils.config import load_config
from Modeling.Src.soilmoist_fl.Data.load import load_splits
from Modeling.Src.soilmoist_fl.Features.preprocess import preprocess_split
from Modeling.Src.soilmoist_fl.Selectors.mi import select_mi
from Modeling.Src.soilmoist_fl.Selectors.elasticnet import select_elasticnet

# Configure import path for dataset_metadata
sys.path.append(str(PROJECT_ROOT / "data" / "splits" / "derived_8.1_pos"))
import dataset_metadata
FEATURE_SET_B = dataset_metadata.OVERALL_SELECTED_FEATURES

Project root: C:\Users\pan\Documents\GitHub\MDR-Project


In [2]:
# Setup Feature Set A (MDR-v25 baseline)
FEATURE_SET_A = [
    'SMAP_sm_pm_interp_ema02',
    'V_rollmin_LST_modis_kobs30',
    'D_sin_DOY', 'G_rain_sum_3d',
    'V_ema_G_API_kobs7',
    'V_rollmin_G_API_kobs30',
    'G_rain_sum_7d',
    'C_lag_LST_modis_kobs30',
    'C_lag_G_API_kobs1',
    'V_ema_G_API_kobs14',
    'V_rollmean_G_API_kobs14',
    'G_API', 'G_DSLR',
    'SMAP_ampm_diff_interp',
    'V_rollmax_G_API_kobs30',
    'V_ema_G_API_kobs30',
    'V_rollmean_s2_b11_kobs7',
    'V_ema_LST_modis_kobs7',
    'V_rollmean_G_API_kobs7',
    'C_lag_s2_b11_kobs30',
    'A_d_E_SAR_diff_kobs14',
    'C_lag_LST_modis_kobs6',
    'A_d_LST_modis_kobs14',
    'A_d_SMAP_sm_interp_kobs14',
    'V_rollstd_SMAP_sm_interp_kobs30',
    'SMAP_sm_interp_grad7',
    'year_frac', 'sin_year', 'cos_year',
    'API_x_year', 'SMAP_x_year',
    'slope', 'elev', 'K_slope_sin',
    'K_slope_cos', 'K_aspect_cos',
    'J_clay_wfrac_b0', 'J_sand_wfrac_b0'
]

# Set up XGBoost parameters using CUDA GPU acceleration
XGB_PARAMS_W = dict(
    objective="reg:pseudohubererror",
    random_state=42,
    n_jobs=-1,
    subsample=0.9,
    colsample_bytree=0.8,
    max_depth=8,
    min_child_weight=2,
    n_estimators=5500,
    learning_rate=0.04,
    reg_lambda=1.5,
    reg_alpha=0.03,
    gamma=0.0,
    device="cuda",
)

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    err = y_true - y_pred
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    ubrmse = np.sqrt(np.mean(((y_true - np.mean(y_true)) - (y_pred - np.mean(y_pred))) ** 2))
    bias = np.mean(err)
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        pearson = float("nan")
    else:
        pearson = np.corrcoef(y_true, y_pred)[0, 1]
    return {"R2": r2, "RMSE": rmse, "ubRMSE": ubrmse, "Bias": bias, "MAE": mae, "Pearson": pearson}

In [3]:
# Load configs and dataset splits
config_path = "notebooks/experiment/derived_8.1_pos-feature-selection/config.yaml"
cfg = load_config(config_path)

loaded = load_splits(cfg)
fold = loaded.folds[0]

target = cfg["data"]["target"]
id_cols = list(cfg["data"].get("id_cols", []) or [])
time_col = cfg["data"].get("time_col")
drop_cols = list(id_cols)
if time_col:
    drop_cols.append(time_col)
    
X_tr_fs, y_tr_fs, _, _ = preprocess_split(fold.train, target, drop_cols=drop_cols)
print(f"Features preprocess complete. Shape: {X_tr_fs.shape}")

coerce_numeric: replaced 1874 inf values with NaN


Features preprocess complete. Shape: (15964, 496)


In [4]:
# Feature Selection: Run Configurations C, D, and E
# Configuration C: ElasticNet selection directly on all 496 features (No MI)
print("\n--- Running Selection C: No MI Filter ---")
enet_c = select_elasticnet(X_tr_fs, y_tr_fs, k=40)
feature_set_c = enet_c["selected"]
print(f"Set C selection completed: {len(feature_set_c)} features selected")

# Configuration D: High MI threshold (k=300) before ElasticNet
print("\n--- Running Selection D: High MI Filter (k=300) ---")
mi_d = select_mi(X_tr_fs, y_tr_fs, k=300)
X_mi_d = X_tr_fs[mi_d["selected"]]
enet_d = select_elasticnet(X_mi_d, y_tr_fs, k=40)
feature_set_d = enet_d["selected"]
print(f"Set D selection completed: {len(feature_set_d)} features selected")

# Configuration E: Hybrid/Bypass Selection
print("\n--- Running Selection E: Hybrid/Bypass ---")
# Bypass MI for static/seasonal/precip features
bypass_cols = [c for c in X_tr_fs.columns if c.startswith('J_') or c.startswith('K_') or c.startswith('D_') or c.startswith('G_') or 'year' in c or c in ['longitude', 'latitude', 'elev', 'slope', 'aspect', 'DOY', 'precip_mm']]
ts_cols = [c for c in X_tr_fs.columns if c not in bypass_cols]

# MI on time-series features only
mi_out = select_mi(X_tr_fs[ts_cols], y_tr_fs, k=100)
selected_ts = mi_out["selected"]

# Combine selected time-series features with bypass features
candidate_cols = selected_ts + bypass_cols
print(f"Candidate features count: {len(candidate_cols)} ({len(selected_ts)} TS + {len(bypass_cols)} Bypass)")

# ElasticNet selection on Candidates directly (k=40)
enet_e = select_elasticnet(X_tr_fs[candidate_cols], y_tr_fs, k=40)
feature_set_e = enet_e["selected"]
print(f"Set E selection completed: {len(feature_set_e)} features selected")


--- Running Selection C: No MI Filter ---


Set C selection completed: 40 features selected

--- Running Selection D: High MI Filter (k=300) ---


c:\Users\pan\Documents\GitHub\MDR-Project\notebooks\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.678e-02, tolerance: 1.418e-02
  model = cd_fast.enet_coordinate_descent_gram(


c:\Users\pan\Documents\GitHub\MDR-Project\notebooks\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.375e-02, tolerance: 1.513e-02
  model = cd_fast.enet_coordinate_descent_gram(


c:\Users\pan\Documents\GitHub\MDR-Project\notebooks\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.743e-02, tolerance: 1.468e-02
  model = cd_fast.enet_coordinate_descent_gram(


...

c:\Users\pan\Documents\GitHub\MDR-Project\notebooks\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.073e-02, tolerance: 1.468e-02
  model = cd_fast.enet_coordinate_descent_gram(


c:\Users\pan\Documents\GitHub\MDR-Project\notebooks\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.275e-02, tolerance: 1.468e-02
  model = cd_fast.enet_coordinate_descent_gram(


Set D selection completed: 40 features selected

--- Running Selection E: Hybrid/Bypass ---


Candidate features count: 182 (100 TS + 82 Bypass)


Set E selection completed: 40 features selected


In [5]:
# Prepare Data for Modeling and Calculate Recency Weights
train_df = fold.train.copy()
val_df = fold.val.copy()
test_df = fold.test.copy()

for df in [train_df, val_df, test_df]:
    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.month.astype(int)
    df["year"] = df["date"].dt.year.astype(float)
    
trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
y_trainval = np.asarray(trainval_df[target]).ravel()
y_test = np.asarray(test_df[target]).ravel()

# Compute temporal recency weights (Drift) with beta = 0.2
years_tv = trainval_df["year"]
max_year = years_tv.max()
beta = 0.2
w_trainval = np.exp(beta * (years_tv - max_year))
w_trainval = w_trainval / w_trainval.mean()

print(f"Recency weights prepared. Trainval size: {trainval_df.shape[0]} rows, Test size: {test_df.shape[0]} rows")

Recency weights prepared. Trainval size: 23113 rows, Test size: 8902 rows


In [6]:
# Model Training and Performance Evaluation
feature_sets = {
    "A (v25 Baseline)": FEATURE_SET_A,
    "B (Default: MI=120)": FEATURE_SET_B,
    "C (No MI Filter)": feature_set_c,
    "D (High MI: k=300)": feature_set_d,
    "E (Hybrid: Bypass MI)": feature_set_e
}

results = []

for name, f_set in feature_sets.items():
    print(f"\nEvaluating Feature Set: {name} (Size: {len(f_set)})")
    
    # Build training matrix
    X_trainval = trainval_df[f_set].copy()
    X_test = test_df[f_set].copy()
    
    for col in f_set:
        X_trainval[col] = pd.to_numeric(X_trainval[col], errors="coerce")
        X_test[col] = pd.to_numeric(X_test[col], errors="coerce")
        
    model = XGBRegressor(**XGB_PARAMS_W)
    model.fit(X_trainval, y_trainval, sample_weight=w_trainval, verbose=0)
    
    preds = np.asarray(model.predict(X_test)).ravel()
    metrics = compute_metrics(y_test, preds)
    metrics["Feature Set"] = name
    metrics["Size"] = len(f_set)
    results.append(metrics)
    
    print(f"  R2: {metrics['R2']:.4f} | RMSE: {metrics['RMSE']:.4f} | Pearson: {metrics['Pearson']:.4f}")

# Final comparison table
print("\n" + "="*80)
print("FINAL MODEL COMPARISON SUMMARY")
print("="*80)
results_df = pd.DataFrame(results)
results_df = results_df[["Feature Set", "Size", "R2", "RMSE", "ubRMSE", "Bias", "MAE", "Pearson"]]
print(results_df.to_string(index=False, formatters={
    'R2': '{:,.4f}'.format,
    'RMSE': '{:,.4f}'.format,
    'ubRMSE': '{:,.4f}'.format,
    'Bias': '{:+,.4f}'.format,
    'MAE': '{:,.4f}'.format,
    'Pearson': '{:,.4f}'.format
}))
print("="*80)

# Save selected feature lists to JSON
import json
feature_dump = {
    "Set_C": list(feature_set_c),
    "Set_D": list(feature_set_d),
    "Set_E": list(feature_set_e)
}
with open("notebooks/experiment/derived_8.1_pos-feature-selection/selected_features_comparison.json", "w") as f:
    json.dump(feature_dump, f, indent=2)


Evaluating Feature Set: A (v25 Baseline) (Size: 38)


c:\Users\pan\Documents\GitHub\MDR-Project\notebooks\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [22:00:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  R2: 0.6280 | RMSE: 0.0642 | Pearson: 0.8247

Evaluating Feature Set: B (Default: MI=120) (Size: 40)


  R2: 0.4909 | RMSE: 0.0751 | Pearson: 0.7130

Evaluating Feature Set: C (No MI Filter) (Size: 40)


  R2: 0.6343 | RMSE: 0.0637 | Pearson: 0.8289

Evaluating Feature Set: D (High MI: k=300) (Size: 40)


  R2: 0.6595 | RMSE: 0.0614 | Pearson: 0.8270

Evaluating Feature Set: E (Hybrid: Bypass MI) (Size: 40)


  R2: 0.6309 | RMSE: 0.0640 | Pearson: 0.8203

FINAL MODEL COMPARISON SUMMARY
          Feature Set  Size     R2   RMSE ubRMSE    Bias    MAE Pearson
     A (v25 Baseline)    38 0.6280 0.0642 0.0598 -0.0233 0.0491  0.8247
  B (Default: MI=120)    40 0.4909 0.0751 0.0742 -0.0118 0.0550  0.7130
     C (No MI Filter)    40 0.6343 0.0637 0.0596 -0.0224 0.0479  0.8289
   D (High MI: k=300)    40 0.6595 0.0614 0.0595 -0.0155 0.0448  0.8270
E (Hybrid: Bypass MI)    40 0.6309 0.0640 0.0610 -0.0194 0.0477  0.8203
